<a href="https://colab.research.google.com/github/KAMWINEJONAN/AI-Project__spam-detection/blob/main/AI_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import os

# ── Step 1: Load the Dataset ──────────────────────────────────────────────────
# Source: UCI SMS Spam Collection Dataset

# Using the mirror (identical data):
URL = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
df = pd.read_csv(URL, sep="\t", header=None, names=["label", "text"])

# Create the 'data' directory if it doesn't exist
os.makedirs('data', exist_ok=True)

# Save locally so you don't re-download every time
df.to_csv("data/spam.csv", index=False)
print("Dataset saved to data/spam.csv")

# ── Explore ───────────────────────────────────────────────────────────────────
print("\n=== FIRST 5 ROWS ===")
print(df.head())

print("\n=== SHAPE (rows, columns) ===")
print(df.shape)

print("\n=== LABEL COUNTS ===")
print(df['label'].value_counts())

print("\n=== SPAM PERCENTAGE ===")
spam_pct = df['label'].value_counts(normalize=True) * 100
print(spam_pct.round(2))

print("\n=== MISSING VALUES ===")
print(df.isnull().sum())

print("\n=== SAMPLE SPAM EMAIL ===")
print(df[df['label'] == 'spam']['text'].iloc[0])

print("\n=== SAMPLE HAM EMAIL ===")
print(df[df['label'] == 'ham']['text'].iloc[0])

Dataset saved to data/spam.csv

=== FIRST 5 ROWS ===
  label                                               text
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...

=== SHAPE (rows, columns) ===
(5572, 2)

=== LABEL COUNTS ===
label
ham     4825
spam     747
Name: count, dtype: int64

=== SPAM PERCENTAGE ===
label
ham     86.59
spam    13.41
Name: proportion, dtype: float64

=== MISSING VALUES ===
label    0
text     0
dtype: int64

=== SAMPLE SPAM EMAIL ===
Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's

=== SAMPLE HAM EMAIL ===
Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...


In [ ]:
import pandas as pd
import string
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer

# Download required NLTK data (will only download if not already present)
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True) # Added this line to download punkt_tab

# Load Data (assuming it's saved from Step 1)
df = pd.read_csv("data/spam.csv")

# ── Preprocessing Function ────────────────────────────────────────────────────
ps = PorterStemmer()
stop_words = set(stopwords.words('english'))

def preprocess(text):
    # 1. Lowercase
    text = text.lower()

    # 2. Remove punctuation
    # Use str.maketrans to remove punctuation efficiently
    text = text.translate(str.maketrans('', '', string.punctuation))

    # 3. Tokenize — split into list of words
    tokens = word_tokenize(text)

    # 4. Remove stopwords — drop "the", "is", "a", "you" etc.
    tokens = [w for w in tokens if w not in stop_words]

    # 5. Stemming — reduce to root form
    tokens = [ps.stem(w) for w in tokens]

    # 6. Join back into a single string
    return " ".join(tokens)

# ── Apply to Dataset ──────────────────────────────────────────────────────────
df['clean_text'] = df['text'].apply(preprocess)

# ── Show Results ──────────────────────────────────────────────────────────────
print("=== BEFORE vs AFTER PREPROCESSING ===\n")
for i in [2, 5, 10]:   # pick a spam and a couple of ham rows
    print(f"[{df['label'][i].upper()}]")
    print(f"  RAW:   {df['text'][i]}")
    print(f"  CLEAN: {df['clean_text'][i]}")
    print()

# ── Save Cleaned Data ─────────────────────────────────────────────────────────
df.to_csv("data/spam_clean.csv", index=False)
print("Cleaned dataset saved to data/spam_clean.csv")
print(f"Columns now: {list(df.columns)}")

# ── Feature Extraction: TF-IDF Vectorization ──────────────────────────────────
# Initialize TF-IDF Vectorizer
# min_df ignores terms that have a document frequency strictly lower than the given threshold.
# max_df ignores terms that have a document frequency strictly higher than the given threshold.
# max_features builds a vocabulary that only considers the top max_features ordered by term frequency.
vectorizer = TfidfVectorizer(min_df=5, max_df=0.8, max_features=5000)

# Fit and transform the processed text
X = vectorizer.fit_transform(df['clean_text'])

# Get the feature names (words) for later analysis
feature_names = vectorizer.get_feature_names_out()

print(f"\nShape of TF-IDF feature matrix (X): {X.shape}")
print(f"Number of unique words (features): {len(feature_names)}")

# Prepare the target variable (y)
y = df['label'].apply(lambda x: 1 if x == 'spam' else 0) # Convert 'ham'/'spam' to 0/1

print(f"\nShape of target variable (y): {y.shape}")
print("Target variable distribution (0=ham, 1=spam):")
display(y.value_counts())

=== BEFORE vs AFTER PREPROCESSING ===

[SPAM]
  RAW:   Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's
  CLEAN: free entri 2 wkli comp win fa cup final tkt 21st may 2005 text fa 87121 receiv entri questionstd txt ratetc appli 08452810075over18

[SPAM]
  RAW:   FreeMsg Hey there darling it's been 3 week's now and no word back! I'd like some fun you up for it still? Tb ok! XxX std chgs to send, £1.50 to rcv
  CLEAN: freemsg hey darl 3 week word back id like fun still tb ok xxx std chg send £150 rcv

[HAM]
  RAW:   I'm gonna be home soon and i don't want to talk about this stuff anymore tonight, k? I've cried enough today.
  CLEAN: im gon na home soon dont want talk stuff anymor tonight k ive cri enough today

Cleaned dataset saved to data/spam_clean.csv
Columns now: ['label', 'text', 'clean_text']

Shape of TF-IDF feature matrix (X): (5572, 1522)
Number of unique words (features): 

,count
label,
0,4825
1,747


In [ ]:
import string
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer

# Download NLTK stopwords (will only download if not already present)
nltk.download('stopwords', quiet=True)

# Initialize English stop words
stop_words = set(stopwords.words('english'))

# Function for text preprocessing
def preprocess_text(text):
    # 1. Lowercasing
    text = text.lower()
    # 2. Remove punctuation
    text = ''.join([char for char in text if char not in string.punctuation])
    # 3. Remove stop words and tokenize
    words = text.split()
    words = [word for word in words if word not in stop_words]
    return ' '.join(words)

# Apply preprocessing to the 'text' column
df['processed_text'] = df['text'].apply(preprocess_text)

# Display the original and processed text for comparison
print("Original vs. Processed Text Sample:")
display(df[['text', 'processed_text']].head())

# 4. TF-IDF Vectorization
# Initialize TF-IDF Vectorizer
# min_df ignores terms that have a document frequency strictly lower than the given threshold.
# max_df ignores terms that have a document frequency strictly higher than the given threshold.
# max_features builds a vocabulary that only considers the top max_features ordered by term frequency.
vidector = TfidfVectorizer(min_df=5, max_df=0.8, max_features=5000)

# Fit and transform the processed text
X = vidector.fit_transform(df['processed_text'])

# Get the feature names (words) for later analysis
feature_names = vidector.get_feature_names_out()

print(f"\nShape of TF-IDF feature matrix (X): {X.shape}")
print(f"Number of unique words (features): {len(feature_names)}")

# Prepare the target variable (y)
y = df['label'].apply(lambda x: 1 if x == 'spam' else 0) # Convert 'ham'/'spam' to 0/1

print(f"\nShape of target variable (y): {y.shape}")
print("Target variable distribution (0=ham, 1=spam):")
display(y.value_counts())

Original vs. Processed Text Sample:


,text,processed_text
0,"Go until jurong point, crazy.. Available only ...",go jurong point crazy available bugis n great ...
1,Ok lar... Joking wif u oni...,ok lar joking wif u oni
2,Free entry in 2 a wkly comp to win FA Cup fina...,free entry 2 wkly comp win fa cup final tkts 2...
3,U dun say so early hor... U c already then say...,u dun say early hor u c already say
4,"Nah I don't think he goes to usf, he lives aro...",nah dont think goes usf lives around though



Shape of TF-IDF feature matrix (X): (5572, 1644)
Number of unique words (features): 1644

Shape of target variable (y): (5572,)
Target variable distribution (0=ham, 1=spam):


,count
label,
0,4825
1,747


In [ ]:
# 1. Error Analysis: Examine Misclassified Samples
# We'll use the original DataFrame, the test labels (y_test), and the model's predictions (y_pred).

# Ensure that the index of y_test aligns with the original DataFrame's index or use the original df for text retrieval
# First, let's get the indices of the test set from the split (assuming train_test_split was used)
from sklearn.model_selection import train_test_split

# Re-split to get the test indices
X_train, X_test, y_train_idx, y_test_idx = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# We need the original text for error analysis, which corresponds to the indices in y_test_idx
df_test = df.loc[y_test_idx.index]
df_test['original_index'] = df_test.index

# Add actual and predicted labels to the test DataFrame
df_test['actual_label'] = y_test_idx
df_test['predicted_label'] = y_pred

# Convert numerical labels back to 'ham'/'spam' for readability
label_map = {0: 'ham', 1: 'spam'}
df_test['actual_label_str'] = df_test['actual_label'].map(label_map)
df_test['predicted_label_str'] = df_test['predicted_label'].map(label_map)

print("\n=== MISCLASSIFIED SAMPLES ===")

# False Positives (Predicted Spam, Actual Ham)
print("\n--- False Positives (Predicted Spam, Actual Ham) ---")
false_positives = df_test[(df_test['actual_label_str'] == 'ham') & (df_test['predicted_label_str'] == 'spam')]
if not false_positives.empty:
    display(false_positives[['text', 'actual_label_str', 'predicted_label_str']].head(5))
else:
    print("No False Positives found.")

# False Negatives (Predicted Ham, Actual Spam)
print("\n--- False Negatives (Predicted Ham, Actual Spam) ---")
false_negatives = df_test[(df_test['actual_label_str'] == 'spam') & (df_test['predicted_label_str'] == 'ham')]
if not false_negatives.empty:
    display(false_negatives[['text', 'actual_label_str', 'predicted_label_str']].head(5))
else:
    print("No False Negatives found.")

# Correctly Classified Spam
print("\n--- Correctly Classified Spam ---")
correct_spam = df_test[(df_test['actual_label_str'] == 'spam') & (df_test['predicted_label_str'] == 'spam')]
if not correct_spam.empty:
    display(correct_spam[['text', 'actual_label_str', 'predicted_label_str']].head(5))
else:
    print("No Correctly Classified Spam found.")



=== MISCLASSIFIED SAMPLES ===

--- False Positives (Predicted Spam, Actual Ham) ---


,text,actual_label_str,predicted_label_str
554,Ok. Every night take a warm bath drink a cup o...,ham,spam
2304,Should I tell my friend not to come round til ...,ham,spam
5163,Ok leave no need to ask,ham,spam
5213,3 pa but not selected.,ham,spam
5177,Very strange. and are watching the 2nd one n...,ham,spam



--- False Negatives (Predicted Ham, Actual Spam) ---


,text,actual_label_str,predicted_label_str
576,"You have won ?1,000 cash or a ?2,000 prize! To...",spam,ham
3059,You are now unsubscribed all services. Get ton...,spam,ham
2669,Wanna get laid 2nite? Want real Dogging locati...,spam,ham
1122,Do you want 750 anytime any network mins 150 t...,spam,ham
5,FreeMsg Hey there darling it's been 3 week's n...,spam,ham



--- Correctly Classified Spam ---


,text,actual_label_str,predicted_label_str
3585,Hi 07734396839 IBH Customer Loyalty Offer: The...,spam,spam
1460,Bought one ringtone and now getting texts cost...,spam,spam
2070,"Eerie Nokia tones 4u, rply TONE TITLE to 8007 ...",spam,spam
3780,"Claim a 200 shopping spree, just call 08717895...",spam,spam
5030,I'd like to tell you my deepest darkest fantas...,spam,spam


In [ ]:
print("\n--- Full False Positives (Predicted Spam, Actual Ham) ---")
display(false_positives[['text', 'actual_label_str', 'predicted_label_str']])

print("\n--- Full False Negatives (Predicted Ham, Actual Spam) ---")
display(false_negatives[['text', 'actual_label_str', 'predicted_label_str']])


--- Full False Positives (Predicted Spam, Actual Ham) ---


,text,actual_label_str,predicted_label_str
554,Ok. Every night take a warm bath drink a cup o...,ham,spam
2304,Should I tell my friend not to come round til ...,ham,spam
5163,Ok leave no need to ask,ham,spam
5213,3 pa but not selected.,ham,spam
5177,Very strange. and are watching the 2nd one n...,ham,spam
...,...,...,...
4535,I have no money 4 steve mate! !,ham,spam
3148,Oh thats late! Well have a good night and i wi...,ham,spam
5174,Water logging in desert. Geoenvironmental impl...,ham,spam
2396,"Babe, I'm back ... Come back to me ...",ham,spam



--- Full False Negatives (Predicted Ham, Actual Spam) ---


,text,actual_label_str,predicted_label_str
576,"You have won ?1,000 cash or a ?2,000 prize! To...",spam,ham
3059,You are now unsubscribed all services. Get ton...,spam,ham
2669,Wanna get laid 2nite? Want real Dogging locati...,spam,ham
1122,Do you want 750 anytime any network mins 150 t...,spam,ham
5,FreeMsg Hey there darling it's been 3 week's n...,spam,ham
...,...,...,...
1507,Thanks for the Vote. Now sing along with the s...,spam,ham
505,+123 Congratulations - in this week's competit...,spam,ham
2686,URGENT! We are trying to contact U. Todays dra...,spam,ham
2987,Reply to win £100 weekly! What professional sp...,spam,ham


In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import numpy as np

# Initialize and train the Multinomial Naive Bayes model with resampled data
mnb_smote = MultinomialNB()
mnb_smote.fit(X_resampled, y_resampled)

# Make predictions on the original (unresampled) test set
y_pred_smote = mnb_smote.predict(X_test)

# Evaluate the model with SMOTE
accuracy_smote = accuracy_score(y_test, y_pred_smote)
precision_smote = precision_score(y_test, y_pred_smote)
recall_smote = recall_score(y_test, y_pred_smote)
f1_smote = f1_score(y_test, y_pred_smote)
conf_matrix_smote = confusion_matrix(y_test, y_pred_smote)

print("\n=== Model Evaluation (Multinomial Naive Bayes with SMOTE) ===")
print(f"Accuracy: {accuracy_smote:.4f}")
print(f"Precision: {precision_smote:.4f}")
print(f"Recall: {recall_smote:.4f}")
print(f"F1-Score: {f1_smote:.4f}")
print("Confusion Matrix:\n", conf_matrix_smote)

# Compare with original model (assuming original metrics are available as accuracy, precision, recall, f1, conf_matrix)
print("\n=== Comparison to Original Model ===")
print(f"Original Accuracy: {accuracy:.4f} -> New Accuracy: {accuracy_smote:.4f}")
print(f"Original Precision: {precision:.4f} -> New Precision: {precision_smote:.4f}")
print(f"Original Recall: {recall:.4f} -> New Recall: {recall_smote:.4f}")
print(f"Original F1-Score: {f1:.4f} -> New F1-Score: {f1_smote:.4f}")
print("Original Confusion Matrix:\n", conf_matrix)
print("New Confusion Matrix:\n", conf_matrix_smote)



=== Model Evaluation (Multinomial Naive Bayes with SMOTE) ===
Accuracy: 0.7516
Precision: 0.1145
Recall: 0.1275
F1-Score: 0.1206
Confusion Matrix:
 [[819 147]
 [130  19]]

=== Comparison to Original Model ===
Original Accuracy: 0.9803 -> New Accuracy: 0.7516
Original Precision: 1.0000 -> New Precision: 0.1145
Original Recall: 0.8523 -> New Recall: 0.1275
Original F1-Score: 0.9203 -> New F1-Score: 0.1206
Original Confusion Matrix:
 [[966   0]
 [ 22 127]]
New Confusion Matrix:
 [[819 147]
 [130  19]]


In [ ]:
from sklearn.linear_model import LogisticRegression

# Initialize and train the Logistic Regression model with resampled data
# Setting a higher max_iter for convergence, and 'liblinear' solver for good performance on smaller datasets.
# C is the inverse of regularization strength; smaller values specify stronger regularization.
lr_smote = LogisticRegression(solver='liblinear', random_state=42, max_iter=200)
lr_smote.fit(X_resampled, y_resampled)

# Make predictions on the original (unresampled) test set
y_pred_lr_smote = lr_smote.predict(X_test)

# Evaluate the Logistic Regression model with SMOTE
accuracy_lr_smote = accuracy_score(y_test, y_pred_lr_smote)
precision_lr_smote = precision_score(y_test, y_pred_lr_smote)
recall_lr_smote = recall_score(y_test, y_pred_lr_smote)
f1_lr_smote = f1_score(y_test, y_pred_lr_smote)
conf_matrix_lr_smote = confusion_matrix(y_test, y_pred_lr_smote)

print("\n=== Model Evaluation (Logistic Regression with SMOTE) ===")
print(f"Accuracy: {accuracy_lr_smote:.4f}")
print(f"Precision: {precision_lr_smote:.4f}")
print(f"Recall: {recall_lr_smote:.4f}")
print(f"F1-Score: {f1_lr_smote:.4f}")
print("Confusion Matrix:\n", conf_matrix_lr_smote)

# Compare with previous models
print("\n=== Comparison to Previous Models ===")
print("-----------------------------------------------------------------------------------------------------------------")
print("| Metric       | Original MNB | MNB + SMOTE | LR + SMOTE |")
print("-----------------------------------------------------------------------------------------------------------------")
print(f"| Accuracy     | {accuracy:.4f}       | {accuracy_smote:.4f}        | {accuracy_lr_smote:.4f}     |")
print(f"| Precision    | {precision:.4f}       | {precision_smote:.4f}        | {precision_lr_smote:.4f}     |")
print(f"| Recall       | {recall:.4f}       | {recall_smote:.4f}        | {recall_lr_smote:.4f}     |")
print(f"| F1-Score     | {f1:.4f}       | {f1_smote:.4f}        | {f1_lr_smote:.4f}     |")
print("-----------------------------------------------------------------------------------------------------------------")



=== Model Evaluation (Logistic Regression with SMOTE) ===
Accuracy: 0.7596
Precision: 0.1161
Recall: 0.1208
F1-Score: 0.1184
Confusion Matrix:
 [[829 137]
 [131  18]]

=== Comparison to Previous Models ===
-----------------------------------------------------------------------------------------------------------------
| Metric       | Original MNB | MNB + SMOTE | LR + SMOTE |
-----------------------------------------------------------------------------------------------------------------
| Accuracy     | 0.9803       | 0.7516        | 0.7596     |
| Precision    | 1.0000       | 0.1145        | 0.1161     |
| Recall       | 0.8523       | 0.1275        | 0.1208     |
| F1-Score     | 0.9203       | 0.1206        | 0.1184     |
-----------------------------------------------------------------------------------------------------------------


In [ ]:
from sklearn.linear_model import LogisticRegression

# Define the parameter grid for Logistic Regression
param_grid_lr = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'solver': ['liblinear', 'saga'] # 'liblinear' good for small datasets, 'saga' for larger ones and supports more penalties
}

# Initialize GridSearchCV
grid_search_lr = GridSearchCV(
    LogisticRegression(random_state=42, max_iter=200), # Increase max_iter for convergence
    param_grid_lr,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

# Fit GridSearchCV on the SMOTE-resampled training data
grid_search_lr.fit(X_resampled, y_resampled)

# Get the best parameters and best score
best_params_lr = grid_search_lr.best_params_
best_score_lr = grid_search_lr.best_score_

print(f"\nBest parameters for LR: {best_params_lr}")
print(f"Best F1-score on cross-validation (LR): {best_score_lr:.4f}")

# Retrain LR with the best parameters and evaluate on the test set
lr_tuned = LogisticRegression(**best_params_lr, random_state=42, max_iter=200)
lr_tuned.fit(X_resampled, y_resampled)
y_pred_lr_tuned = lr_tuned.predict(X_test)

accuracy_lr_tuned = accuracy_score(y_test, y_pred_lr_tuned)
precision_lr_tuned = precision_score(y_test, y_pred_lr_tuned)
recall_lr_tuned = recall_score(y_test, y_pred_lr_tuned)
f1_lr_tuned = f1_score(y_test, y_pred_lr_tuned)
conf_matrix_lr_tuned = confusion_matrix(y_test, y_pred_lr_tuned)

print("\n=== Model Evaluation (Tuned Logistic Regression with SMOTE) ===")
print(f"Accuracy: {accuracy_lr_tuned:.4f}")
print(f"Precision: {precision_lr_tuned:.4f}")
print(f"Recall: {recall_lr_tuned:.4f}")
print(f"F1-Score: {f1_lr_tuned:.4f}")
print("Confusion Matrix:\n", conf_matrix_lr_tuned)


Fitting 5 folds for each of 12 candidates, totalling 60 fits

Best parameters for LR: {'C': 10, 'solver': 'liblinear'}
Best F1-score on cross-validation (LR): 0.9750

=== Model Evaluation (Tuned Logistic Regression with SMOTE) ===
Accuracy: 0.7525
Precision: 0.1243
Recall: 0.1409
F1-Score: 0.1321
Confusion Matrix:
 [[818 148]
 [128  21]]


In [ ]:
import pandas as pd

# Create a dictionary to store all model results
results = {
    'Model': [
        'Original MNB',
        'MNB + SMOTE',
        'MNB + SMOTE (Tuned)',
        'LR + SMOTE',
        'LR + SMOTE (Tuned)'
    ],
    'Accuracy': [
        accuracy,
        accuracy_smote,
        accuracy_mnb_tuned,
        accuracy_lr_smote,
        accuracy_lr_tuned
    ],
    'Precision': [
        precision,
        precision_smote,
        precision_mnb_tuned,
        precision_lr_smote,
        precision_lr_tuned
    ],
    'Recall': [
        recall,
        recall_smote,
        recall_mnb_tuned,
        recall_lr_smote,
        recall_lr_tuned
    ],
    'F1-Score': [
        f1,
        f1_smote,
        f1_mnb_tuned,
        f1_lr_smote,
        f1_lr_tuned
    ]
}

# Create a DataFrame from the results
results_df = pd.DataFrame(results)

# Display the comparison table
print("\n=== Model Performance Comparison ===")
display(results_df.round(4))

print("\n--- Analysis of Confusion Matrices ---")
print("Original MNB Confusion Matrix:\n", conf_matrix)
print("MNB + SMOTE Confusion Matrix:\n", conf_matrix_smote)
print("MNB + SMOTE (Tuned) Confusion Matrix:\n", conf_matrix_mnb_tuned)
print("LR + SMOTE Confusion Matrix:\n", conf_matrix_lr_smote)
print("LR + SMOTE (Tuned) Confusion Matrix:\n", conf_matrix_lr_tuned)



=== Model Performance Comparison ===


,Model,Accuracy,Precision,Recall,F1-Score
0,Original MNB,0.9803,1.0000,0.8523,0.9203
1,MNB + SMOTE,0.7516,0.1145,0.1275,0.1206
2,MNB + SMOTE (Tuned),0.7525,0.1104,0.1208,0.1154
3,LR + SMOTE,0.7596,0.1161,0.1208,0.1184
4,LR + SMOTE (Tuned),0.7525,0.1243,0.1409,0.1321



--- Analysis of Confusion Matrices ---
Original MNB Confusion Matrix:
 [[966   0]
 [ 22 127]]
MNB + SMOTE Confusion Matrix:
 [[819 147]
 [130  19]]
MNB + SMOTE (Tuned) Confusion Matrix:
 [[821 145]
 [131  18]]
LR + SMOTE Confusion Matrix:
 [[829 137]
 [131  18]]
LR + SMOTE (Tuned) Confusion Matrix:
 [[818 148]
 [128  21]]


In [ ]:
import joblib
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split

# Define the directory to save the assets
model_dir = 'trained_models'
os.makedirs(model_dir, exist_ok=True)

# 1. Save the TfidfVectorizer
joblib.dump(vectorizer, os.path.join(model_dir, 'tfidf_vectorizer.joblib'))
print(f"TfidfVectorizer saved to {model_dir}/tfidf_vectorizer.joblib")

# 2. Retrain the Original Multinomial Naive Bayes model (without SMOTE or tuning)
# We need X_train and y_train that were used for the original MNB model
# Re-split the data to get the original X_train, y_train for training the 'original' model.
# Ensure this split is consistent with the one used for the original evaluation.
X_train_original, X_test_original, y_train_original, y_test_original = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

mnb_original = MultinomialNB()
mnb_original.fit(X_train_original, y_train_original)

# Save the trained Multinomial Naive Bayes model
joblib.dump(mnb_original, os.path.join(model_dir, 'spam_detector_mnb_original.joblib'))
print(f"Original Multinomial Naive Bayes model saved to {model_dir}/spam_detector_mnb_original.joblib")


TfidfVectorizer saved to trained_models/tfidf_vectorizer.joblib
Original Multinomial Naive Bayes model saved to trained_models/spam_detector_mnb_original.joblib


## How to Use the Saved Model for New Predictions

This section demonstrates how you would load your saved `TfidfVectorizer` and `Multinomial Naive Bayes` model to classify new, unseen messages.

In [ ]:
import joblib
import os
import string
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
import nltk

# Ensure NLTK data is downloaded if running in a fresh environment
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Define the preprocessing function (must be the exact same as used for training)
ps = PorterStemmer()
stop_words = set(stopwords.words('english'))

def preprocess_new_message(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = word_tokenize(text)
    tokens = [w for w in tokens if w not in stop_words]
    tokens = [ps.stem(w) for w in tokens]
    return " ".join(tokens)

# Define the directory where the models are saved
model_dir = 'trained_models'

# Load the saved TfidfVectorizer and model
try:
    loaded_vectorizer = joblib.load(os.path.join(model_dir, 'tfidf_vectorizer.joblib'))
    loaded_model = joblib.load(os.path.join(model_dir, 'spam_detector_mnb_original.joblib'))
    print("Model and Vectorizer loaded successfully!")
except FileNotFoundError:
    print(f"Error: Model or Vectorizer files not found in '{model_dir}'. Please ensure they were saved correctly.")
    loaded_vectorizer = None
    loaded_model = None

if loaded_vectorizer and loaded_model:
    # Example new messages
    new_messages = [
        "How are you?",
        "Congratulations! You've won a free iPhone. Click here to claim.",
        "Hey, just checking in. How are you doing?",
        "URGENT! Your account has been suspended. Verify here: http://bit.ly/maliciouslink",
        "Can we meet tomorrow at 10 AM?"
    ]

    print("\n=== Classifying New Messages ===")
    for message in new_messages:
        # Preprocess the new message
        clean_message = preprocess_new_message(message)

        # Transform the cleaned message using the loaded vectorizer
        transformed_message = loaded_vectorizer.transform([clean_message])

        # Make a prediction
        prediction = loaded_model.predict(transformed_message)

        # Interpret the prediction
        label = "SPAM" if prediction[0] == 1 else "HAM"
        print(f"Message: '{message}'\nClassification: {label}\n")


Model and Vectorizer loaded successfully!

=== Classifying New Messages ===
Message: 'How are you?'
Classification: HAM

Message: 'Congratulations! You've won a free iPhone. Click here to claim.'
Classification: SPAM

Message: 'Hey, just checking in. How are you doing?'
Classification: HAM

Message: 'URGENT! Your account has been suspended. Verify here: http://bit.ly/maliciouslink'
Classification: SPAM

Message: 'Can we meet tomorrow at 10 AM?'
Classification: HAM

